# TMDB 데이터 수집

In [17]:
import os
import sys

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [18]:
import os
from pathlib import Path

import requests

current_dir = Path.cwd()
project_root = current_dir.parent.parent if "notebooks" in str(current_dir) else current_dir
env_path = project_root / ".env"
TMDB_API_KEY = os.getenv("TMDB-API-KEY")

In [53]:
from data_scraping.common.ml_data_loader import load_links_data_ml
from data_scraping.common.tmdb_loader import load_tmdb_data

tmdb_data = load_tmdb_data()
links_data = load_links_data_ml()

In [66]:
imdb_to_tmdb = dict(zip(links_data["imdb_id"], links_data["tmdb_id"]))

In [ ]:
import pandas as pd


def get_movie_credits(movie_id: int, language: str = "ko-KR"):
    """
    TMDb API를 사용하여 영화의 출연진 및 제작진 정보를 가져옵니다.

    Args:
        movie_id: TMDb 영화 ID
        language: 언어 코드 (기본값: ko-KR)

    Returns:
        출연진 및 제작진 정보 딕셔너리 (cast, crew 포함)
    """
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    params = {"api_key": TMDB_API_KEY, "language": language}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ 출연진/제작진 정보 요청 실패: {e}")
        return None


# 예제: 영화 ID로 출연진/제작진 정보 가져오기
movie_id = imdb_to_tmdb["tt0120338"]
credits = get_movie_credits(movie_id)
cast_df = pd.DataFrame(credits.get("cast"))
cast_df["tmdb_id"] = "tt0120338"
cast_df

,adult,gender,id,known_for_department,name,original_name,popularity,profile_path,cast_id,character,credit_id,order,tmdb_id
0,False,2,6193,Acting,레오나르도 디카프리오,Leonardo DiCaprio,6.5327,/wo2hJpn04vbtmh0B9utCFdsQhxM.jpg,412,Jack Dawson,675afdf316336674f49cfb45,0,tt0120338
1,False,1,204,Acting,케이트 윈슬렛,Kate Winslet,5.3637,/e3tdop3WhseRnn8KwMVLAV25Ybv.jpg,413,Rose DeWitt Bukater,675afdffc7d3f2f93e131ebc,1,tt0120338
2,False,2,1954,Acting,빌리 제인,Billy Zane,2.7608,/wr4fuwLzQvW1G0MS7cmQ3ObFjvL.jpg,414,Cal Hockley,675afe15deb41dde9ded0ea2,2,tt0120338
3,False,1,8534,Acting,캐시 베이츠,Kathy Bates,2.4266,/nP7zRmXJyMyj2OohclSSo6JAEbD.jpg,415,Molly Brown,675afe29d9d5195fadaa30e9,3,tt0120338
4,False,1,3713,Acting,프랜시스 피셔,Frances Fisher,2.9363,/3iNDgd54IIj8g8hGqhhUjM6TeWd.jpg,416,Ruth DeWitt Bukater,675afe3416336674f49cfb6e,4,tt0120338
...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,False,2,93214,Directing,Steven Quale,Steven Quale,1.1835,/5egEoUCfyEMxjznMwYU27PbTrkn.jpg,198,Engine Room Crewman (uncredited),5880d193c3a368312c00bd74,111,tt0120338
112,False,0,1741742,Lighting,R. Gern Trowbridge,R. Gern Trowbridge,0.0776,None,202,Drowning Man (uncredited),5880d24b92514134be00c1ed,112,tt0120338
113,False,1,11670,Acting,Olivia Rosewood,Olivia Rosewood,0.8341,/nstMN0354sEfWUC7C8xKnBJ6z3v.jpg,199,Mary Marvin (uncredited),5880d1b092514134be00c145,113,tt0120338
114,False,2,1741740,Acting,John Slade,John Slade,0.1609,None,200,Ohio Man (uncredited),5880d22692514134b800ca53,114,tt0120338


In [70]:
import sys

import pandas as pd

# 여러 영화의 cast 정보를 담을 리스트
all_cast_data = []

total_movies = len(imdb_to_tmdb)
print("영화 캐스팅 정보 수집 시작...")

for i, (imdb_id, tmdb_id) in enumerate(imdb_to_tmdb.items(), 1):
    print(f"\r{i}/{total_movies} 진행 중...", end="")  # 같은 줄에 덮어쓰기

    credits = get_movie_credits(tmdb_id)

    if credits and "cast" in credits:
        cast_list = credits.get("cast", [])

        # 각 출연진 정보에 tmdb_id 추가
        for cast_member in cast_list:
            cast_member["tmdb_id"] = tmdb_id
            all_cast_data.append(cast_member)

print()  # 진행 상황 이후 개행

# 전체 데이터를 DataFrame으로 변환
cast_df = pd.DataFrame(all_cast_data)

# 결과 확인
print(f"총 {len(cast_df)} 개의 출연진 정보")
cast_df.head()

영화 캐스팅 정보 수집 시작...
231/87585 진행 중...❌ 출연진/제작진 정보 요청 실패: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Read timed out. (read timeout=None)
597/87585 진행 중...❌ 출연진/제작진 정보 요청 실패: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/538286.0/credits?api_key=d0bdfd19f47e664400c67871424183c4&language=ko-KR
707/87585 진행 중...❌ 출연진/제작진 정보 요청 실패: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie//credits?api_key=d0bdfd19f47e664400c67871424183c4&language=ko-KR
716/87585 진행 중...❌ 출연진/제작진 정보 요청 실패: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie//credits?api_key=d0bdfd19f47e664400c67871424183c4&language=ko-KR
755/87585 진행 중...❌ 출연진/제작진 정보 요청 실패: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie//credits?api_key=d0bdfd19f47e664400c67871424183c4&language=ko-KR
776/87585 진행 중...❌ 출연진/제작진 정보 요청 실패: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie//credits?api_key=d0bdfd19f47e664400c6

,adult,gender,id,known_for_department,name,original_name,popularity,profile_path,cast_id,character,credit_id,order,tmdb_id
0,False,2,31,Acting,톰 행크스,Tom Hanks,9.2821,/eKF1sGJRrZJbfBG1KirPt1cfNd3.jpg,14,Woody (voice),52fe4284c3a36847f8024f95,0,862.0
1,False,2,12898,Acting,팀 앨런,Tim Allen,2.7198,/woWhZzFILVhYMAvsPL171HjMY0y.jpg,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,1,862.0
2,False,2,7167,Acting,돈 리클리스,Don Rickles,1.2474,/iJLQV4dcbTUgxlWJakjDldzlMXS.jpg,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,862.0
3,False,2,12899,Acting,짐 바니,Jim Varney,0.7413,/zvBFmvKUrPvE6FW35O3RP4i1ZPp.jpg,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,3,862.0
4,False,2,12900,Acting,월리스 숀,Wallace Shawn,3.2246,/wVaM1WlFKDce4esThwL4XtNLhOe.jpg,18,Rex (voice),52fe4284c3a36847f8024fa5,4,862.0


In [ ]:
# cast_df.to_csv('cast_data.csv', index=False)

In [72]:
cast_df

,adult,gender,id,known_for_department,name,original_name,popularity,profile_path,cast_id,character,credit_id,order,tmdb_id
0,False,2,31,Acting,톰 행크스,Tom Hanks,9.2821,/eKF1sGJRrZJbfBG1KirPt1cfNd3.jpg,14,Woody (voice),52fe4284c3a36847f8024f95,0,862.0
1,False,2,12898,Acting,팀 앨런,Tim Allen,2.7198,/woWhZzFILVhYMAvsPL171HjMY0y.jpg,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,1,862.0
2,False,2,7167,Acting,돈 리클리스,Don Rickles,1.2474,/iJLQV4dcbTUgxlWJakjDldzlMXS.jpg,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,862.0
3,False,2,12899,Acting,짐 바니,Jim Varney,0.7413,/zvBFmvKUrPvE6FW35O3RP4i1ZPp.jpg,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,3,862.0
4,False,2,12900,Acting,월리스 숀,Wallace Shawn,3.2246,/wVaM1WlFKDce4esThwL4XtNLhOe.jpg,18,Rex (voice),52fe4284c3a36847f8024fa5,4,862.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1707759,False,2,1219282,Acting,Murray MacLeod,Murray MacLeod,0.3187,None,28,Johnny Taylor,63125a23ba131b007c54d051,4,182776.0
1707760,False,1,69103,Acting,Lori Martin,Lori Martin,0.6858,/gCSg0IoP6z5xyOvEZDGwhh6uAsP.jpg,29,Diane Patton,63125a3702842000830cd0fd,5,182776.0
1707761,False,1,160762,Acting,Melody Patterson,Melody Patterson,0.2566,/AnnjqHwDjV3omvl4cJwRbIcODjz.jpg,30,April Banner,63125a46d7a70a007e1c5e3a,6,182776.0
1707762,False,2,592447,Acting,우엘리 슈테크,Ueli Steck,0.2852,/etgUHwUYdcSTivJyCGn2fOFTp0y.jpg,1,,64f85ec44ccc501867e973ed,0,1174725.0


In [73]:
cast_df["known_for_department"].unique()

array(['Acting', 'Production', 'Writing', 'Directing',
       'Costume & Make-Up', 'Art', 'Crew', 'Sound', 'Lighting', 'Editing',
       'Camera', 'Creator', 'Visual Effects', None], dtype=object)